# Clustering Application (Album Indexing)

In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
sys.path.append(os.getcwd())

In [3]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [4]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)["reid_project"]

In [ ]:
from src.utils.clustering import perform_clustering, evaluate_clusters, visualize_cluster_accuracy, calculate_cluster_purity
from src.utils.evaluator import evaluate, compute_distmat, extract_features
from src.dataloaders.market_dataset import MarketDataset
from src.models.resnet50 import ResNet50

## 1. Clustering pipeline

Extract embeddings from all person crops, cluster with DBSCAN or Agglomerative Clustering on cosine distance.

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet50(
    num_classes=config['market1501']['num_classes'], 
    feature_dim=config["models"]["resnet50"]["feature_dim"],
    last_stride=config["models"]["resnet50"]["last_stride"]
)
model_path = "results/exp_resnet50_v1/resnet50_reid_epoch_1.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

ResNet50(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64

In [7]:
transform = transforms.Compose([
    transforms.Resize(config['market1501']['img_size'], 
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
gallery_ds = MarketDataset(root_dir=config['market1501']['root_dir'], subset='test', transform=transform)
gallery_loader = DataLoader(gallery_ds, batch_size=32, shuffle=False)

In [ ]:
g_feat, g_pids, g_camids = extract_features(model, gallery_loader, device)
features = g_feat.cpu().numpy()

Extraction:  99%|█████████▉| 407/410 [18:10<00:51, 17.00s/it]  

In [ ]:
pred_labels = perform_clustering(features, eps=0.35)

In [ ]:
visualize_cluster_accuracy(0, pred_labels, g_pids, gallery_ds)

In [ ]:
avg_purity = calculate_cluster_purity(g_pids, pred_labels)
print(f"Albums average purity: {avg_purity:.2%}")

## 2. Threshold Sensitivity

In [ ]:
eps_range = np.linspace(0.1, 0.7, 15)
results = []

for e in eps_range:
    labels = perform_clustering(features, eps=e)
    metrics = evaluate_clusters(g_pids, labels)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    results.append({'eps': e, 'NMI': metrics['NMI'], 'clusters': n_clusters})

import pandas as pd
import seaborn as sns
df_sens = pd.DataFrame(results)
sns.lineplot(data=df_sens, x='eps', y='NMI')
plt.title("Impact of Distance Threshold (eps) on Grouping Quality")

## 3. Album Demo

In [ ]:
def display_album(cluster_id, pred_labels, dataset, max_imgs=10):
    indices = np.where(pred_labels == cluster_id)[0]
    plt.figure(figsize=(15, 3))
    for i, idx in enumerate(indices[:max_imgs]):
        img, _, _ = dataset[idx]
        img = img.permute(1, 2, 0).numpy()
        plt.subplot(1, max_imgs, i+1)
        plt.imshow(img)
        plt.axis('off')
    plt.suptitle(f"Album Cluster {cluster_id} ({len(indices)} images)")
    plt.show()

unique_clusters, counts = np.unique(pred_labels[pred_labels != -1], return_counts=True)
for c in unique_clusters[np.argsort(counts)[-5:]]:
    display_album(c, pred_labels, gallery_ds)